#Prepare the CDC batches 
###Converts the openFDA snapshot into a CDC event feed, one file per year
1. Read json file
2. Initial event (report_date)
3. Status Event
4. Controlled Errors for evaluation
5. Write into 'staging'

In [0]:
from pyspark.sql import functions as F

In [0]:
RAW_FILE = "/Volumes/workspace/food_safety/raw/food-enforcement-0001-of-0001.json"
STAGING  = "/Volumes/workspace/food_safety/staging"

###Step 1. Read JSON file

In [0]:
raw_df = spark.read.option("multiLine", True).json(RAW_FILE)
recalls = (raw_df
    .select(F.explode("results").alias("r"))   # one row for each recall
    .select("r.*")
    .drop("openfda"))                          # blank object in food
print("Recalls on the snapshot:", recalls.count())
 
def has_date(c):
    # True si la columna trae una fecha YYYYMMDD
    return F.length(F.coalesce(F.col(c), F.lit(""))) == 8


###Step 2. Initial Event

In [0]:
initial = (recalls
    .withColumn("event_type", F.lit("INITIAL"))
    .withColumn("event_ts", F.expr(
        "to_timestamp(to_date(report_date, 'yyyyMMdd')) + INTERVAL 1 SECOND"))
    # If it got terminated later, when published it was still open
    .withColumn("status", F.when(has_date("termination_date"), F.lit("Ongoing"))
                           .otherwise(F.col("status")))
    # When published, termination date was still not defined
    .withColumn("termination_date", F.lit(""))) 

###Step 3. Status Changes event

In [0]:
terminated = (recalls
    .filter(has_date("termination_date"))
    .withColumn("event_type", F.lit("STATUS_CHANGE"))
    .withColumn("event_ts", F.expr(
        "to_timestamp(to_date(termination_date, 'yyyyMMdd')) + INTERVAL 2 SECONDS"))
    .withColumn("status", F.lit("Terminated")))
 
events = (initial.unionByName(terminated)
          .withColumn("batch_year", F.year("event_ts"))
          .filter(F.col("batch_year").isNotNull()))


###Step 4. Controlled Errors to evaluate quality standards

In [0]:
recent = events.filter(F.col("batch_year") >= 2020)
bad_id  = (recent.sample(fraction=0.003, seed=42)
           .withColumn("recall_number", F.lit(None).cast("string")))
bad_cls = (recent.sample(fraction=0.002, seed=7)
           .withColumn("classification", F.lit("Class X")))
events_final = events.unionByName(bad_id).unionByName(bad_cls)


###Step 5. Write a single JSON file per year into staging volume

In [0]:
years = sorted(r.batch_year for r in events_final.select("batch_year").distinct().collect())
for y in years:
    tmp = f"{STAGING}/tmp_{y}"
    (events_final.filter(F.col("batch_year") == y).drop("batch_year")
        .coalesce(1).write.mode("overwrite").json(tmp))
    part = [f.path for f in dbutils.fs.ls(tmp) if f.name.startswith("part-")][0]
    dbutils.fs.cp(part, f"{STAGING}/recall_events_{y}.json")
    dbutils.fs.rm(tmp, recurse=True)
 
print("Lotes preparados:", years)
display(events_final.groupBy("batch_year", "event_type").count()
        .orderBy("batch_year", "event_type"))
